# 04 — Model Comparison

Compares all trained models across poverty thresholds using MAE, RMSE, and R².

> **Note:** Thresholds `$3` and `$4.20` show degenerate metrics (R² ≈ −0.06, RMSE in trillions)  
> and are excluded from visualisations pending investigation. They are labelled **⚠ to be checked**.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from config import *
from model_pipeline import load_model_bundle, predict_ssp

## 1. Load results

In [ ]:
results = pd.read_csv(OUTPUTS_DIR / "model_comparison_by_threshold.csv")

# Separate valid thresholds from broken ones
BROKEN_THRESHOLDS = ["$3", "$4.20"]
VALID_THRESHOLDS  = [t for t in POVERTY_THRESHOLDS if t not in BROKEN_THRESHOLDS]

valid   = results[results["threshold"].isin(VALID_THRESHOLDS)].copy()
broken  = results[results["threshold"].isin(BROKEN_THRESHOLDS)].copy()

print(f"Thresholds included : {VALID_THRESHOLDS}")
print(f"Thresholds excluded : {BROKEN_THRESHOLDS}  ⚠ to be checked (R² < 0)")
print(f"\nRows in valid set   : {len(valid)}")
print(f"Rows in broken set  : {len(broken)}")

## 2. Full comparison table

In [ ]:
display_cols = ["threshold", "model_name", "rmse", "mae", "r2"]

table_valid = (
    valid[display_cols]
    .sort_values(["threshold", "rmse"])
    .reset_index(drop=True)
)

# Append broken thresholds as a labelled block
broken_display = broken[["threshold", "model_name"]].copy()
broken_display["rmse"] = "⚠ to be checked"
broken_display["mae"]  = "⚠ to be checked"
broken_display["r2"]   = "⚠ to be checked"

full_table = pd.concat([table_valid, broken_display], ignore_index=True)
print(full_table.to_string(index=False))

## 3. Bar charts — RMSE, MAE, R² per threshold

In [ ]:
metrics = [
    ("rmse", "RMSE (pp)",  "#d62728", False),
    ("mae",  "MAE (pp)",   "#1f77b4", False),
    ("r2",   "R²",         "#2ca02c", True),
]

fig, axes = plt.subplots(
    len(metrics), len(VALID_THRESHOLDS),
    figsize=(5 * len(VALID_THRESHOLDS), 4 * len(metrics)),
    sharey="row"
)

for row, (metric, ylabel, color, higher_better) in enumerate(metrics):
    for col, threshold in enumerate(VALID_THRESHOLDS):
        ax  = axes[row][col]
        sub = valid[valid["threshold"] == threshold].sort_values(metric)
        if higher_better:
            sub = sub.sort_values(metric, ascending=False)

        names  = sub["model_name"].str.replace("_", "\n")
        values = sub[metric].values
        bars   = ax.bar(names, values, color=color, alpha=0.75)

        # Highlight best bar
        best_idx = 0
        bars[best_idx].set_edgecolor("gold")
        bars[best_idx].set_linewidth(2.5)

        ax.set_title(f"{threshold}/day", fontsize=9, fontweight="bold")
        ax.set_ylabel(ylabel if col == 0 else "", fontsize=8)
        ax.tick_params(axis="x", labelsize=7)
        ax.grid(True, linestyle="--", alpha=0.4, axis="y")

# Add note about excluded thresholds
note = f"⚠  Thresholds {BROKEN_THRESHOLDS} excluded — degenerate metrics (R² < 0). To be checked."
fig.text(0.5, -0.01, note, ha="center", fontsize=8, color="#d62728")

plt.suptitle("Model comparison — RMSE / MAE / R²  (gold border = best per threshold)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "model_comparison_04.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Best model per threshold

In [ ]:
best_per_threshold = (
    valid.loc[valid.groupby("threshold")["rmse"].idxmin()]
    [["threshold", "model_name", "rmse", "mae", "r2"]]
    .reset_index(drop=True)
)

print("Best model per threshold (lowest test RMSE):")
print("=" * 60)
print(best_per_threshold.to_string(index=False))

# Append broken thresholds
broken_best = pd.DataFrame({
    "threshold":  BROKEN_THRESHOLDS,
    "model_name": "⚠ to be checked",
    "rmse":        float("nan"),
    "mae":         float("nan"),
    "r2":          float("nan"),
})
print()
print(broken_best.to_string(index=False))